# Convolutional Neural Networks

Convolutional neural networks (CNNs) are the standard architecture for working with images. A plain fully connected network treats every pixel as an independent input, which throws away the fact that nearby pixels belong together and forces the model to learn a huge number of weights. A CNN instead slides small learnable filters across the image, so it can detect a useful pattern (an edge, a corner, a texture) wherever it appears and with far fewer parameters.

This notebook builds up the intuition for the core operations (convolution, stride, padding, pooling) and then trains a small CNN on the MNIST digits dataset with Keras.

## Learning Objectives

At the end of this notebook, you should be able to:

- Explain how convolution, stride, padding, and pooling transform an image.
- Build a convolutional neural network in Keras using `Conv2D`, `MaxPooling2D`, `Flatten`, and `Dense` layers.
- Train the model on MNIST and read its accuracy and loss curves to spot overfitting.
- Evaluate the trained model with predictions and a confusion matrix.

## How a Convolution Works

A convolution slides a small matrix of weights (a **kernel** or **filter**) across the image. The kernel is much smaller than the image, so the same weights are reused at every position. That weight sharing is what makes CNNs efficient: one filter that detects a vertical edge is useful anywhere in the image, so we do not need a separate set of weights per pixel.

### Convolution

To convolve a kernel with an image:

- Slide the kernel (a small matrix of weights) over the 2D image.
- At each position, multiply each kernel weight by the pixel it overlaps (element-wise).
- Sum those products into a single output pixel.

<img src="https://miro.medium.com/max/1400/1*Fw-ehcNBR9byHtho-Rxbtw.gif" alt="Drawing" style="width: 400px;"/>

### Stride

The **stride** is the step size the kernel moves by between positions. A stride of 1 moves one pixel at a time, a stride of 2 skips every other position (which also shrinks the output). The animation below shows a stride of 2.

<img src="https://miro.medium.com/max/588/1*BMngs93_rm2_BpJFH2mS0Q.gif" alt="Drawing" style="width: 300px;"/>

We convolve the filter with the image by sliding the 3x3 filter over the 6x6 image. At each position we multiply the overlapping pixels, sum them, and store the sum as the output pixel for that position. We then move the filter one cell to the right and repeat. When the row is finished we move down to the next row, and so on.

Let's calculate a couple of values together, then you will do the rest on your own.

![](assets/Convolution.gif)

Now continue on your own until the filter has covered the whole original image.

You can check the result in `assets/convolved.jpeg`.

**Q**: What is the size of the resulting image, if the original image is `n x n` and the filter is `f x f`?

**A**: `n-f+1`

You just ran your first edge-detection algorithm.

That is, in very simple terms, how convolutional neural networks work. The difference is that we do not choose the filter values by hand: the filters are the parameters (the weights, your `w`s) that the network learns during training.

### Padding

**Padding** adds extra pixels around the edges of the image before convolving. We saw that the output is smaller than the input, and that edge pixels are visited by the filter fewer times than central pixels, which is not always desirable. Padding the image so the output keeps its size addresses both issues.

<img src="https://miro.medium.com/max/790/1*1okwhewf5KCtIPaFib4XaA.gif" alt="Drawing" style="width: 300px;"/>

How much padding should we use? Two common options:

- `valid`: no padding.
- `same`: pad so the output keeps the same size as the input.

**Q**: What is the size of the resulting image, if the original image is `n x n`, the filter is `f x f`, and we use `p` pixels of padding on each side?

**A**: `n-f+1+2p`

**Q**: What is the size of the resulting image, if the original image is `n x n`, the filter is `f x f`, we use padding of `p`, and a stride of `s`?

**A**: `(n-f+2p)/s + 1`

### Pooling

As well as convolutional layers, CNNs use **pooling layers**. Pooling accumulates (pools) the features produced by a convolutional layer and reduces the spatial size of the data.

**Max pooling**, the most common method, keeps only the highest value in each pooling window.

![max_pooling.png](assets/max_pooling.png)

Another option is **average pooling**, which keeps the average value of each window instead of the maximum.

### Colour (RGB) Images

For a colour (RGB) image we start with three filters, one per colour channel, convolve each channel as before, and then sum the three results into a single output image (here 4x4x1).

### One Layer of a CNN

A CNN layer usually applies more than one filter. The third dimension of the output equals the number of filters used, so each filter produces its own feature map.

![rgb.png](assets/rgb.png)

In CNNs we generally use more than one filter per layer, so the depth of the output equals the number of filters.

![rgb_2.png](assets/rgb_2.png)

In CNN diagrams this is usually drawn like this:

![one_layer.png](assets/one_layer.png)

### Refresher: Image Kernels

Spend some time with this interactive explanation of [image kernels](https://setosa.io/ev/image-kernels/) (feel free to play with all the settings) and discuss:

- What does the number at each pixel location in a greyscale image mean?
- What is a kernel?
- How is a kernel applied to an image?
- How does changing the numbers in the kernel change the output image?
- What features of an image can we highlight with a kernel?

### Example CNN Architecture

A typical CNN stacks a few convolutional layers (often alternating with pooling layers), then flattens the result and finishes with one or more **fully connected** (dense) layers that do the final classification. Shown below is one of the classic architectures, LeNet-5.

The diagram below shows the same idea as a flow: the image passes through alternating convolution and pooling layers that extract features, is flattened into a vector, and is classified by dense layers.

```mermaid
flowchart LR
    A[Input image] --> B[Conv + ReLU]
    B --> C[Max pooling]
    C --> D[Conv + ReLU]
    D --> E[Max pooling]
    E --> F[Flatten]
    F --> G[Dense layers]
    G --> H[Class probabilities]
```

![cnn_with_pooling.png](assets/cnn_with_pooling.png)

Using filters has two benefits:

- It lowers the number of parameters, because the same filter is reused across the whole image.
- It makes sense, because a filter that detects a particular feature is useful in many areas of an image.

Nearby pixels are related, and this matters:

- Flattening the image straight away would lose this spatial information.
- Distant pixels are mostly unrelated, so we do not need to connect every input pixel to every output pixel.

### Visualising a CNN

Explore this interactive 3D visualisation of a CNN classifying digits: https://adamharley.com/nn_vis/cnn/2d.html

### Other Layers to Reduce Overfitting

- **Batch normalisation**: makes networks faster and more stable by re-centring and re-scaling the inputs to each layer.
- **Dropout**: a regularisation technique that randomly drops units during training to prevent complex co-adaptations, which acts like averaging over many networks.

## Building a CNN with Keras

We now put the theory into practice: we load the MNIST handwritten-digit dataset and train a small CNN to classify the digits 0 to 9.

In [ ]:
# importing the required modules
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Dense, MaxPooling2D, Flatten

from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
import numpy as np

### 0. Load the Dataset

In [ ]:
# splitting mnist data present within keras dataset into train and test
(xtrain, ytrain), (xtest, ytest) = mnist.load_data()

Check the shapes of the arrays.

In [ ]:
xtrain.shape, xtest.shape, ytrain.shape, ytest.shape

The result, an explanation: the training set has 60,000 images and the test set 10,000, each a 28x28 greyscale image, with one label per image.

#### Reshape to (images, width, height, channels)

A 2D convolution expects each image to have three dimensions: width, height, and number of colour channels. Colour images have 3 channels (RGB); a greyscale image has 1, so we add a trailing channel dimension.

In [ ]:
xtrain.reshape(60000, 28, 28, 1).shape

In [ ]:
xtrain = xtrain.reshape(60000, 28, 28, 1)
xtest = xtest.reshape(10000, 28, 28, 1)

MNIST has 10 classes, the digits 0 to 9:

In [ ]:
# From the mnist dataset we have 10 classes to categorize, ie numbers from 0-9
np.unique(ytrain)

#### One-hot Encode the Labels

The labels are single integers (for example `3`), but the softmax output layer produces a probability per class, so we convert each label into a 10-element vector with a 1 in the position of the correct class.

In [ ]:
# creates a copy of ytest, for more details (https://docs.python.org/3/library/copy.html)
# we do this because we want to keep the original labels and another variable with the labels encoded
ytest_true = ytest.copy()

In [ ]:
to_categorical(ytest)  # eg. of how to one-hot-encode categorical variables

In [ ]:
ytrain = to_categorical(ytrain)
ytest = to_categorical(ytest)

In [ ]:
ytest

### 1. Define the Model

We build the network as a `Sequential` stack of layers. The design mirrors the LeNet-style architecture above: two convolution-and-pooling blocks to extract features, then a `Flatten` and a `Dense` softmax layer to classify into 10 digits.

- A `Conv2D` layer with 6 filters, ReLU activation, then `MaxPooling2D`.
- A `Conv2D` layer with 16 filters, ReLU activation, then `MaxPooling2D`.
- `Flatten` to turn the feature maps into a vector.
- A `Dense` layer with 10 neurons and softmax activation, one output per class.

We use `padding='same'` so the convolutions keep the spatial size, and let pooling do the downsampling.

In [ ]:
# creating the tensorflow model with the above definition
from tensorflow.keras import backend as K

K.clear_session()  # clear the cache of model parameters
model = Sequential(
    [
        Conv2D(
            filters=6,
            kernel_size=(3, 3),
            strides=(1, 1),
            padding="same",
            activation="relu",
            input_shape=(28, 28, 1),
        ),
        MaxPooling2D(pool_size=(2, 2), strides=2),
        Conv2D(
            filters=16,
            kernel_size=(3, 3),
            strides=(1, 1),
            padding="same",
            activation="relu",
        ),
        MaxPooling2D(pool_size=(2, 2), strides=2),
        Flatten(),
        Dense(units=10, activation="softmax"),
    ]
)

#### Layer reference

- [Conv2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D)
- [MaxPooling2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/MaxPooling2D)
- [Flatten](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Flatten)
- [Dense](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense)

The number of units in the final `Dense` layer is the number of classes. For binary classification use a `sigmoid` activation in the last layer; for multi-class classification use `softmax`.

For the intermediate layers, [ReLU](https://www.tensorflow.org/api_docs/python/tf/keras/activations/relu) is the recommended default for both multilayer perceptrons and CNNs. See the full list of [activations](https://www.tensorflow.org/api_docs/python/tf/keras/activations).

- [General guidelines for building CNNs and tuning hyperparameters](https://medium.com/data-science/a-guide-to-an-efficient-way-to-build-neural-network-architectures-part-ii-hyper-parameter-42efca01e5d7)
- [Hyperparameter tuning with Keras Tuner](https://www.tensorflow.org/tutorials/keras/keras_tuner)

Check the model summary.

In [ ]:
model.summary()

The result, an explanation: the model has only **8,790 trainable parameters**. That is tiny for an image classifier, and it shows the payoff of weight sharing: the convolutional layers reuse the same small filters across the whole image instead of learning a separate weight per pixel.

### 2. Compile the Model

Compiling tells Keras how to train: which optimiser updates the weights, which loss to minimise, and which metric to report. We use `categorical_crossentropy` because the labels are one-hot encoded across 10 classes, and track `accuracy`.

In [ ]:
model.compile(
    optimizer="rmsprop", loss="categorical_crossentropy", metrics=["accuracy"]
)

#### Compilation reference

- [Optimizers](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers): the [Adam](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam) optimiser is a popular, effective default, and a low learning rate (for example 0.001 or 0.0001) can improve performance.
- [Losses](https://www.tensorflow.org/api_docs/python/tf/keras/losses): use `binary_crossentropy` for binary classification, `categorical_crossentropy` for multi-class, and an error metric such as `mean_squared_error` for regression.
- [Metrics](https://www.tensorflow.org/api_docs/python/tf/keras/metrics).

### 3. Fit the Model

We now train for 10 epochs with a batch size of 128, holding out 30% of the training data for validation.

- The number of **epochs** is how many complete passes the model makes through the training data. You typically stop before the model overfits, or once the validation loss stops improving.
- The **batch size** is how many samples the model processes before it updates its weights. It is usually a power of 2 for computational efficiency.

In [ ]:
epochs = 10
accuracy_metrics = model.fit(
    xtrain, ytrain, batch_size=128, epochs=epochs, validation_split=0.3
)

The result, an explanation: after 10 epochs the model reaches around 99.3% training accuracy and about 97.9% validation accuracy. The validation accuracy lagging slightly behind the training accuracy is normal and a mild sign that the model has started to fit the training set a little more closely than the validation set.

### 4. Inspect the Training Curves

Plotting accuracy and loss per epoch shows whether the model is still improving and whether it is starting to overfit (training metric keeps improving while the validation metric stalls or worsens).

In [ ]:
from matplotlib import pyplot as plt

plt.plot(accuracy_metrics.history["accuracy"], label="train_accuracy")
plt.plot(accuracy_metrics.history["val_accuracy"], label="val_accuracy")
plt.xlabel("epochs")
plt.ylabel("accuracy")
plt.legend()

In [ ]:
print(
    f"At the end of the {epochs}th epoch the validation accuracy has reached {'{:.4f}'.format(accuracy_metrics.history['val_accuracy'][-1])}"
)

The result, an explanation: the printed validation accuracy at the final epoch is about 0.9788, so the model classifies roughly 98 out of every 100 unseen digits correctly.

In [ ]:
from matplotlib import pyplot as plt

plt.plot(accuracy_metrics.history["loss"], label="train_loss")
plt.plot(accuracy_metrics.history["val_loss"], label="val_loss")
plt.xlabel("epochs")
plt.ylabel("loss")
plt.legend()

In [ ]:
print(
    f"At the end of the {epochs}th epoch the validation loss has decreased to {'{:.4f}'.format(accuracy_metrics.history['val_loss'][-1])}"
)

The result, an explanation: the validation loss at the final epoch is about 0.0930. The training loss keeps falling, and if the validation loss starts to rise while the training loss falls, that gap is the signature of overfitting.

#### Reading the Curves for Overfitting

![Example of overfitting](https://i.stack.imgur.com/FkPkr.png)

Overfitting is best judged from the validation **loss** rather than accuracy, because accuracy is not always a reliable measure of a classifier's performance. In the example plot above, the validation loss clearly rises as training continues, while the training loss keeps falling: a classic overfitting pattern.

Let's look at one image from the test set and check what the model predicts for it.

In [ ]:
# visualization of one image from the test dataset
plt.imshow(xtest[8])

This is the one-hot encoded true label for that image:

In [ ]:
# original encoded label corresponding to the test data
ytest[8]

In [ ]:
xtest.shape

Predict the class probabilities for the whole test set:

In [ ]:
# predict your test data
pred = model.predict(xtest)

In [ ]:
pred

`pred` gives the probability of each of the 10 classes for every image in `xtest`.

In [ ]:
pred[8].argmax()

The result, an explanation: the highest probability is at index 5, so the model predicts the digit **5** for `xtest[8]`. The true label below is also 5, so this prediction is correct.

In [ ]:
ytest_true[8]  # the true label for test data

In [ ]:
np.unique(ytrain)

### Confusion Matrix

A confusion matrix shows, for each true digit, how the model's predictions were distributed across all classes. The diagonal counts the correct predictions; off-diagonal cells reveal which digits get confused with each other.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

plt.figure(figsize=(8, 8))
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true=ytest_true, y_pred=np.argmax(pred, axis=-1))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(ytest_true))
disp.plot()
plt.show()

The result, an explanation: almost all the mass sits on the diagonal, matching the ~97.9% validation accuracy. The few off-diagonal entries point to the digit pairs the model occasionally mixes up (for example 4 with 9, or 3 with 5), which is where extra training or regularisation would help most.

### Exercise

- Vary the hyperparameters and re-run the model.
- Try changing the number of filters, `kernel_size`, `padding`, `activation`, `pool_size`, `units`, and so on.
- Add more `Conv2D` and `Dense` layers.
- Try different activation functions in the `Conv2D` layers.
- Try different optimisers when compiling the model.

### Bonus: Visualise the Feature Maps

A trained CNN is not a black box: we can look at what each filter responds to. Here we rebuild a small model that outputs the activations right after the first `Conv2D` layer, then plot the 6 feature maps for one digit. Each map highlights a different low-level pattern (for example an edge in a particular direction).

In [ ]:
from tensorflow.keras.models import Model
from numpy import expand_dims

All the layers in the trained model:

In [ ]:
# All the layers present in the created model
model.layers

In [ ]:
model.inputs

Redefine the model so it outputs the activations right after the first hidden layer:

In [ ]:
# redefine model to output right after the first hidden layer
K.clear_session()
model_small = Model(inputs=model.inputs, outputs=model.layers[0].output)
model_small.summary()

Load one image with the required shape:

In [ ]:
# load the image with the required shape
img = xtrain[2]
plt.imshow(img.reshape(28, 28), cmap=plt.cm.Greys)

In [ ]:
img.shape

Expand the dimensions so the image represents a single sample (number of images, width, height, channels):

In [ ]:
# expand dimensions so that it represents a single 'sample' (number of images,width,height,number of channels)
img = expand_dims(img, axis=0)
# img = expand_dims(img, axis=3)
img.shape

Get the feature maps from the first hidden layer:

In [ ]:
# get feature map for first hidden layer
feature_maps = model_small.predict(img)

Plot all 6 feature maps. Each one is the output of a different filter in the first convolutional layer:

In [ ]:
# plot all 6 maps in an 3*2 squares
height = 3
width = 2
ix = 1
plt.figure(figsize=(12, 4))
for _ in range(height):
    for _ in range(width):
        # specify subplot and turn of axis
        ax = plt.subplot(height, width, ix)
        ax.set_xticks([])
        ax.set_yticks([])
        # plot filter channel in grayscale
        plt.imshow(feature_maps[0, :, :, ix - 1], cmap="gray")
        ix += 1
# show the figure
plt.show()

If you observe overfitting while training, try some regularisation:

- [BatchNormalization](https://keras.io/api/layers/normalization_layers/batch_normalization/): add a batch normalisation layer between CNN layers.
- [Dropout](https://keras.io/api/layers/regularization_layers/dropout/): add a dropout layer between dense layers.
- [Callbacks](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks): utilities run at given stages of training. They can help prevent overfitting, visualise progress, save checkpoints, and generate logs. For example [EarlyStopping](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping) stops training when a monitored metric stops improving: `EarlyStopping(monitor='val_loss', patience=5)`. Pass it via `model.fit(..., callbacks=[callback])`.

## Summary

In this notebook you:

- Explained how convolution, stride, padding, and pooling transform an image, and why filters share weights.
- Built a small CNN in Keras with two convolution-and-pooling blocks, a `Flatten`, and a softmax `Dense` layer (only 8,790 parameters).
- Trained it on MNIST, reaching about 97.9% validation accuracy, and read the accuracy and loss curves to reason about overfitting.
- Evaluated the model with single predictions, a confusion matrix, and a look at the first-layer feature maps.

## References & Further Reading

- [**Keras Conv2D layer**](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D): the API for the convolutional layer used here.
- [**Keras Sequential model guide**](https://www.tensorflow.org/guide/keras/sequential_model): how to stack layers into a model.
- [**Image Kernels, explained visually**](https://setosa.io/ev/image-kernels/): an interactive look at what a kernel does to an image.
- [**Intuitively understanding convolutions for deep learning**](https://medium.com/data-science/intuitively-understanding-convolutions-for-deep-learning-1f6f42faee1): a visual, intuition-first walkthrough of convolutions.